# PCA with SVD — pure NumPy

**Dataset:** `breast_cancer.csv` (569 rows × 30 features, binary diagnosis).

PCA is not a classifier. It finds directions of maximum variance; the diagnosis column is only used afterwards to ask whether those directions happen to separate the groups.

## 1. Load and standardise on all rows for this descriptive lab

In [3]:
import numpy as np

raw = np.genfromtxt('breast_cancer.csv', delimiter=',', names=True)
feature_names = list(raw.dtype.names[:-1])
X = np.column_stack([raw[name] for name in feature_names]).astype(float)
y = np.asarray(raw['diagnosis'], dtype=int)

mean = X.mean(axis=0)
scale = X.std(axis=0)
scale[scale == 0] = 1.0
Xs = (X - mean) / scale
print('rows:', X.shape[0], '| features:', X.shape[1])

rows: 569 | features: 30

## 2. SVD, variance ratios and the number of components

In [5]:
centered = Xs - Xs.mean(axis=0)
U, singular_values, Vt = np.linalg.svd(centered, full_matrices=False)
variance = (singular_values ** 2) / (len(Xs) - 1)
ratio = variance / variance.sum()
cumulative = np.cumsum(ratio)

for index in range(5):
    print(f'PC{index + 1}: {ratio[index]:.3f} variance | cumulative {cumulative[index]:.3f}')
print('components for 90% variance:', int(np.searchsorted(cumulative, 0.90) + 1))
print('components for 95% variance:', int(np.searchsorted(cumulative, 0.95) + 1))

PC1: 0.443 variance | cumulative 0.443
PC2: 0.190 variance | cumulative 0.632
PC3: 0.094 variance | cumulative 0.726
PC4: 0.066 variance | cumulative 0.792
PC5: 0.055 variance | cumulative 0.847
components for 90% variance: 7
components for 95% variance: 10

## 3. Project to two dimensions and inspect the class means

In [7]:
components = Vt[:2].T
projected = centered @ components
print('projected shape:', projected.shape)
for label, name in ((0, 'malignant'), (1, 'benign')):
    print(name, 'PC1 mean:', round(float(projected[y == label, 0].mean()), 2),
          '| PC2 mean:', round(float(projected[y == label, 1].mean()), 2))

projected shape: (569, 2)
malignant PC1 mean: -3.71 | PC2 mean: 0.58
benign PC1 mean: 2.21 | PC2 mean: -0.35

## 4. Reconstruction error measures what the projection lost

In [9]:
reconstructed = projected @ components.T + Xs.mean(axis=0)
rmse = float(np.sqrt(np.mean((Xs - reconstructed) ** 2)))
print('2-component reconstruction RMSE:', round(rmse, 3))
print('original feature RMSE:', round(float(np.sqrt(np.mean((Xs - Xs.mean(axis=0)) ** 2))), 3))

2-component reconstruction RMSE: 0.606
original feature RMSE: 1.0

## 5. Limitations

- PCA maximises variance, not class separation. A low-variance direction can still be the useful one.
- Scaling changes the answer. Here every feature was standardised; a raw-unit PCA would be dominated by `mean_area` and `worst_area`.
- The components are linear combinations and can be hard to name. Check loadings before writing a story.
- This lab standardised on all rows for description. In a predictive pipeline, fit the scaler and PCA on training rows only.

**Exercise:** print the five features with the largest absolute loading on PC1. Then explain why a high loading is not a causal claim.